In [1]:
# master module
import pandas as pd
from pathlib import Path


PROJECT_ROOT = Path("/")
RAW_ROOT = PROJECT_ROOT / "UgandaLSMS"
OUT_ROOT = PROJECT_ROOT / "Finished sections"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

WAVE_PATHS = {
    1: RAW_ROOT / "Wave_1",
    2: RAW_ROOT / "Wave_2",
    3: RAW_ROOT / "Wave_3",
    4: RAW_ROOT / "Wave_4",
    5: RAW_ROOT / "Wave_5",
    7: RAW_ROOT / "Wave_7",
    8: RAW_ROOT / "Wave_8",
}


def load_dta(path):
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.strip().str.upper()
    return df


def find_file_case_insensitive(folder, filename):

    filename_upper = filename.upper()

    matches = [p for p in folder.iterdir() if p.name.upper() == filename_upper]

    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find {filename} in {folder}")

    if len(matches) > 1:
        raise ValueError(f"Multiple matches for {filename} in {folder}: {matches}")

    return matches[0]

def first_nonmissing(series):
    nonmissing = series.dropna()
    if len(nonmissing) == 0:
        return pd.NA
    return nonmissing.iloc[0]


def count_yes(series):
    s = series.astype("string").str.strip().str.upper()
    return s.isin(["YES", "Y", "1", "1.0", "TRUE"]).sum()


def sum_numeric(series):
    return pd.to_numeric(series, errors="coerce").sum(min_count=1)




def add_agsec3_labor_aggregates(df_raw, source):

    wave = source["wave"]
    visit = source["visit"]
    df = df_raw.copy()

    if wave in [4, 5]:
        prefix = "A3AQ" if visit == 1 else "A3BQ"

        person_cols = [f"{prefix}33{letter}" for letter in ["A", "B", "C", "D", "E"]]
        day_cols = [f"{prefix}33{letter}_1" for letter in ["A", "B", "C", "D", "E"]]

        available_person_cols = [c for c in person_cols if c in df.columns]
        available_day_cols = [c for c in day_cols if c in df.columns]

        if available_person_cols:
            df["AGG_FAM_LABOUR_COUNT"] = df[available_person_cols].notna().sum(axis=1)
        else:
            df["AGG_FAM_LABOUR_COUNT"] = pd.NA

        if available_day_cols:
            df["AGG_FAM_LABOUR_DAYS"] = (
                df[available_day_cols]
                .apply(pd.to_numeric, errors="coerce")
                .sum(axis=1, min_count=1)
            )
        else:
            df["AGG_FAM_LABOUR_DAYS"] = pd.NA

        return df

    if wave == 8:
        labor_file = source.get("labor_file")

        if labor_file is None:
            df["AGG_FAM_LABOUR_COUNT"] = pd.NA
            df["AGG_FAM_LABOUR_DAYS"] = pd.NA
            return df

        folder = WAVE_PATHS[wave]
        labor_path = find_file_case_insensitive(folder, labor_file)

        df_labor = load_dta(labor_path)

        key_cols = ["HHID", "PARCELID", "PLTID"]

        missing_main_keys = [c for c in key_cols if c not in df.columns]
        missing_labor_keys = [c for c in key_cols if c not in df_labor.columns]

        if missing_main_keys:
            raise KeyError(f"Wave 8 AGSEC3 main file missing keys: {missing_main_keys}")

        if missing_labor_keys:
            raise KeyError(f"Wave 8 AGSEC3 labour file missing keys: {missing_labor_keys}")

        worked_col = "S3AQ33" if visit == 1 else "S3BQ33"
        days_col = "S3AQ33_1" if visit == 1 else "S3BQ33_1"

        if worked_col not in df_labor.columns:
            df_labor[worked_col] = pd.NA

        if days_col not in df_labor.columns:
            df_labor[days_col] = pd.NA

        worked_clean = df_labor[worked_col].astype("string").str.strip().str.upper()

        df_labor["_WORKED_YES"] = worked_clean.isin(
            ["YES", "Y", "1", "1.0", "TRUE"]
        ).astype(int)

        df_labor["_DAYS_NUM"] = pd.to_numeric(df_labor[days_col], errors="coerce")

        labor_agg = (
            df_labor
            .groupby(key_cols, dropna=False)
            .agg(
                AGG_FAM_LABOUR_COUNT=("_WORKED_YES", "sum"),
                AGG_FAM_LABOUR_DAYS=("_DAYS_NUM", lambda x: x.sum(min_count=1)),
            )
            .reset_index()
        )

        df = df.merge(labor_agg, on=key_cols, how="left")

        return df

    return df

def build_source_variable_map(spec, source):
    source_col = source["source_col"]
    variable_map = {}

    for row in spec["crosswalk_rows"]:
        raw = row.get(source_col)

        if raw is None or str(raw).strip() == "":
            continue

        raw = str(raw).strip().upper()

        if row.get("is_key"):
            target = row["standard_name"].strip().upper()
        else:
            target = row["target"].strip().upper()

        variable_map[raw] = target

    return variable_map


def get_expected_cols(spec):
    expected_cols = []

    for row in spec["crosswalk_rows"]:
        if row.get("is_key"):
            expected_cols.append(row["standard_name"].strip().upper())
        else:
            expected_cols.append(row["target"].strip().upper())

    return list(dict.fromkeys(expected_cols))


def get_metadata_cols(spec):
    metadata_cols = ["WAVE", "SOURCE_FILE", "SOURCE_SECTION"]

    extra_metadata = spec.get("metadata_cols", [])

    for col in extra_metadata:
        col = col.upper()
        if col not in metadata_cols:
            metadata_cols.append(col)

    return metadata_cols


def normalize_yes_no_value(x):

    if pd.isna(x):
        return pd.NA

    s = str(x).strip().upper()

    if s in ["", ".", "NAN", "NONE", "<NA>"]:
        return pd.NA

    if s in ["1", "1.0", "YES", "Y", "TRUE"]:
        return "YES"

    if s in ["2", "2.0", "NO", "N", "FALSE", "0", "0.0"]:
        return "NO"

    return x


def filled_to_yes_no(series):

    s = series.astype("string").str.strip()

    is_filled = (
        s.notna()
        & ~s.isin(["", ".", "nan", "NaN", "None", "<NA>"])
    )

    return is_filled.map({True: "YES", False: "NO"})


def postprocess_agsec10(df, source):

    wave = source["wave"]

    # Wave 7/8 special rule:
    # H9Q4A is not directly a Yes/No variable;
    # if it contains anything, code as YES.
    if wave in [7, 8] and "A10Q3" in df.columns:
        df["A10Q3"] = filled_to_yes_no(df["A10Q3"])

    # Normalize regular binary variables.
    # A10Q3 is included for earlier waves.
    binary_cols = ["A10Q3", "A10Q5A", "A10Q5B", "A10Q5C"]

    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].map(normalize_yes_no_value)

    return df


def standardize_one_source(source, spec):
    wave = source["wave"]
    filename = source["file"]
    source_section = source["source_section"]
    variable_map = source["variable_map"]

    folder = WAVE_PATHS[wave]
    path = find_file_case_insensitive(folder, filename)

    df_raw = load_dta(path)

    # Keep AGSEC3 special preprocessing alive
    if spec["section"] == "AGSEC3" and "add_agsec3_labor_aggregates" in globals():
        df_raw = add_agsec3_labor_aggregates(df_raw, source)

    available_map = {
        raw: target
        for raw, target in variable_map.items()
        if raw in df_raw.columns
    }

    missing_raw = [
        raw
        for raw in variable_map
        if raw not in df_raw.columns
    ]

    df = df_raw[list(available_map.keys())].copy()
    df = df.rename(columns=available_map)

    duplicated_cols = df.columns[df.columns.duplicated()].tolist()
    if duplicated_cols:
        raise ValueError(
            f"Duplicate standardized columns in wave {wave}, "
            f"{source_section}: {duplicated_cols}"
        )

    expected_cols = get_expected_cols(spec)

    for col in expected_cols:
        if col not in df.columns:
            df[col] = pd.NA

    if spec["section"] == "AGSEC10":
        df = postprocess_agsec10(df, source)

    for key in spec["standard_keys"]:
        if key in df.columns:
            df[key] = df[key].astype("string").str.strip()

    df.insert(0, "SOURCE_SECTION", source_section)
    df.insert(0, "SOURCE_FILE", path.name)
    df.insert(0, "WAVE", wave)

    for meta_col in spec.get("metadata_cols", []):
        meta_col_upper = meta_col.upper()
        source_key = meta_col.lower()

        if source_key in source:
            df[meta_col_upper] = source[source_key]
        elif meta_col in source:
            df[meta_col_upper] = source[meta_col]
        else:
            df[meta_col_upper] = pd.NA

    metadata_cols = get_metadata_cols(spec)
    final_cols = metadata_cols + expected_cols
    df = df[final_cols]

    print(f"\nLoaded Wave {wave}: {path.name}")
    print(f"Source section: {source_section}")
    print(f"Rows: {len(df):,}")
    print(f"Mapped variables found: {len(available_map):,}/{len(variable_map):,}")

    if missing_raw:
        print(f"Missing raw variables: {missing_raw}")

    return df

def stack_section(spec):
    section_name = spec["section"]
    pieces = []

    for source in spec["sources"]:
        try:
            df_source = standardize_one_source(source, spec)
            pieces.append(df_source)
        except FileNotFoundError as e:
            print(f"\nSKIPPED missing file: {e}")

    if not pieces:
        raise ValueError(f"No files loaded for {section_name}")

    df_stacked = pd.concat(pieces, ignore_index=True)

    print(f"\n==============================")
    print(f"STACKED {section_name}")
    print(f"Rows: {len(df_stacked):,}")
    print(f"Columns: {len(df_stacked.columns):,}")
    print("==============================")

    metadata_cols = get_metadata_cols(spec)

    group_cols = [
        c for c in metadata_cols
        if c in df_stacked.columns and c not in ["SOURCE_FILE", "SOURCE_SECTION"]
    ]

    if group_cols:
        print("\nRows by metadata:")
        print(
            df_stacked
            .groupby(group_cols, dropna=False)
            .size()
            .reset_index(name="ROWS")
            .to_string(index=False)
        )

    keys = spec["standard_keys"]
    dup_subset = group_cols + keys

    duplicate_count = df_stacked.duplicated(subset=dup_subset).sum()

    print(f"\nUnique metadata-key rows: {df_stacked[dup_subset].drop_duplicates().shape[0]:,}")
    print(f"Duplicates on metadata-keys: {duplicate_count:,}")

    return df_stacked

def export_section(df, section_name):
    section_name = section_name.upper()

    parquet_path = OUT_ROOT / f"{section_name}_standardized.parquet"
    csv_path = OUT_ROOT / f"{section_name}_standardized.csv"
    excel_path = OUT_ROOT / f"{section_name}_standardized_preview.xlsx"

    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)

    preview_n = min(len(df), 10000)
    df.head(preview_n).to_excel(excel_path, index=False)

    print(f"\nExported:")
    print(f"Parquet: {parquet_path}")
    print(f"CSV:     {csv_path}")
    print(f"Excel preview first {preview_n:,} rows: {excel_path}")

    return parquet_path, csv_path, excel_path






In [4]:
#AGSEC10 SPEC map


AGSEC10_SPEC = {
    "section": "AGSEC10",
    "level": "extension_source",
    "standard_keys": ["HHID", "EXT_SOURCE_ID"],

    "sources": [
        {"wave": 1, "file": "AGSEC10.dta", "source_section": "AGSEC10", "source_col": "W1"},

        {"wave": 2, "file": "AGSEC9.dta", "source_section": "AGSEC9", "source_col": "W2"},
        {"wave": 3, "file": "AGSEC9.dta", "source_section": "AGSEC9", "source_col": "W3"},
        {"wave": 4, "file": "AGSEC9.dta", "source_section": "AGSEC9", "source_col": "W4"},

        {"wave": 5, "file": "AGSEC9A.dta", "source_section": "AGSEC9A", "source_col": "W5"},
        {"wave": 7, "file": "AGSEC9A.dta", "source_section": "AGSEC9A", "source_col": "W7"},
        {"wave": 8, "file": "AGSEC9A.dta", "source_section": "AGSEC9A", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1": "HHID",
            "W2": "HHID",
            "W3": "HHID",
            "W4": "HHID",
            "W5": "HHID",
            "W7": "HHID",
            "W8": "HHID",
        },
        {
            "target": "A10Q2",
            "is_key": True,
            "standard_name": "EXT_SOURCE_ID",
            "W1": "A10Q2",
            "W2": "A9Q2",
            "W3": "A9Q2",
            "W4": "A9Q2",
            "W5": "A9Q2",
            "W7": "SOURCE_ID",
            "W8": "SOURCE_ID",
        },
        {
            "target": "A10Q3",
            "is_key": False,
            "W1": "A10Q3",
            "W2": "A9Q3",
            "W3": "A9Q3",
            "W4": "A9Q3",
            "W5": "A9Q3",
            "W7": "H9Q04A",
            "W8": "H9Q04A",
        },
        {
            "target": "A10Q5A",
            "is_key": False,
            "W1": "A10Q5A",
            "W2": "A9Q5A",
            "W3": "A9Q5A",
            "W4": "A9Q5A",
            "W5": "A9Q5A",
            "W7": "H9Q05AG__1",
            "W8": "H9Q05AG__1",
        },
        {
            "target": "A10Q5B",
            "is_key": False,
            "W1": "A10Q5B",
            "W2": "A9Q5B",
            "W3": "A9Q5B",
            "W4": "A9Q5B",
            "W5": "A9Q5B",
            "W7": "H9Q05AG__2",
            "W8": "H9Q05AG__2",
        },
        {
            "target": "A10Q5C",
            "is_key": False,
            "W1": "A10Q5C",
            "W2": "A9Q5C",
            "W3": "A9Q5C",
            "W4": "A9Q5C",
            "W5": "A9Q5C",
            "W7": "H9Q05AG__3",
            "W8": "H9Q05AG__3",
        },
    ],
}

In [5]:
#AGSEC10 export mod

for source in AGSEC10_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(AGSEC10_SPEC, source)

agsec10 = stack_section(AGSEC10_SPEC)
export_section(agsec10, "AGSEC10")

agsec10.head()



Loaded Wave 1: AGSEC10.dta
Source section: AGSEC10
Rows: 11,970
Mapped variables found: 6/6

Loaded Wave 2: AGSEC9.dta
Source section: AGSEC9
Rows: 538
Mapped variables found: 6/6

Loaded Wave 3: AGSEC9.dta
Source section: AGSEC9
Rows: 13,617
Mapped variables found: 6/6

Loaded Wave 4: AGSEC9.dta
Source section: AGSEC9
Rows: 14,919
Mapped variables found: 6/6

Loaded Wave 5: AGSEC9A.dta
Source section: AGSEC9A
Rows: 16,140
Mapped variables found: 6/6

Loaded Wave 7: AGSEC9A.dta
Source section: AGSEC9A
Rows: 298
Mapped variables found: 6/6

Loaded Wave 8: AGSEC9A.dta
Source section: AGSEC9A
Rows: 234
Mapped variables found: 6/6

STACKED AGSEC10
Rows: 57,716
Columns: 9

Rows by metadata:
 WAVE  ROWS
    1 11970
    2   538
    3 13617
    4 14919
    5 16140
    7   298
    8   234

Unique metadata-key rows: 57,715
Duplicates on metadata-keys: 1

Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\AGSEC10_standardized.parquet
CSV:     C:\Users\Carl\Desktop\CSB_project

,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,EXT_SOURCE_ID,A10Q3,A10Q5A,A10Q5B,A10Q5C
0,1,AGSEC10.dta,AGSEC10,411300060106,,NO,NaN,NaN,NaN
1,1,AGSEC10.dta,AGSEC10,211100030705,NAADS,NO,NaN,NaN,NaN
2,1,AGSEC10.dta,AGSEC10,211100030705,INPUT SUPPLIER,NO,NaN,NaN,NaN
3,1,AGSEC10.dta,AGSEC10,211100030705,NGO,NO,NaN,NaN,NaN
4,1,AGSEC10.dta,AGSEC10,211100030705,COOPERATIVE,NO,NaN,NaN,NaN


In [ ]:
# AGSEC5 SPEC map



AGSEC5_SPEC = {
    "section": "AGSEC5",
    "level": "crop_plot",
    "standard_keys": ["HHID", "PARCEL_ID", "PLOT_ID", "CROP_ID"],

    "sources": [
        {"wave": 1, "visit": 1, "file": "AGSEC5A.dta", "source_section": "AGSEC5A", "source_col": "W1_A"},
        {"wave": 1, "visit": 2, "file": "AGSEC5B.dta", "source_section": "AGSEC5B", "source_col": "W1_B"},

        {"wave": 2, "visit": 1, "file": "AGSEC5A.dta", "source_section": "AGSEC5A", "source_col": "W2_A"},
        {"wave": 2, "visit": 2, "file": "AGSEC5B.dta", "source_section": "AGSEC5B", "source_col": "W2_B"},

        {"wave": 3, "visit": 1, "file": "AGSEC5A.dta", "source_section": "AGSEC5A", "source_col": "W3_A"},
        {"wave": 3, "visit": 2, "file": "AGSEC5B.dta", "source_section": "AGSEC5B", "source_col": "W3_B"},

        {"wave": 4, "visit": 1, "file": "AGSEC5A.dta", "source_section": "AGSEC5A", "source_col": "W4_A"},
        {"wave": 4, "visit": 2, "file": "AGSEC5B.dta", "source_section": "AGSEC5B", "source_col": "W4_B"},

        {"wave": 5, "visit": 1, "file": "AGSEC5A.dta", "source_section": "AGSEC5A", "source_col": "W5_A"},
        {"wave": 5, "visit": 2, "file": "AGSEC5B.dta", "source_section": "AGSEC5B", "source_col": "W5_B"},

        {"wave": 7, "visit": 1, "file": "AGSEC5A.dta", "source_section": "AGSEC5A", "source_col": "W7_A"},
        {"wave": 7, "visit": 2, "file": "AGSEC5B.dta", "source_section": "AGSEC5B", "source_col": "W7_B"},

        {"wave": 8, "visit": 1, "file": "AGSEC5A.dta", "source_section": "AGSEC5A", "source_col": "W8_A"},
        {"wave": 8, "visit": 2, "file": "AGSEC5B.dta", "source_section": "AGSEC5B", "source_col": "W8_B"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1_A": "HHID", "W1_B": "HHID",
            "W2_A": "HHID", "W2_B": "HHID",
            "W3_A": "HHID", "W3_B": "HHID",
            "W4_A": "HHID", "W4_B": "HHID",
            "W5_A": "HHID", "W5_B": "HHID",
            "W7_A": "HHID", "W7_B": "HHID",
            "W8_A": "HHID", "W8_B": "HHID",
        },
        {
            "target": "A5AQ1",
            "is_key": True,
            "standard_name": "PARCEL_ID",
            "W1_A": "A5AQ1", "W1_B": "A5BQ1",
            "W2_A": "PRCID", "W2_B": "PRCID",
            "W3_A": "PARCELID", "W3_B": "PARCELID",
            "W4_A": "PARCELID", "W4_B": "PARCELID",
            "W5_A": "PARCELID", "W5_B": "PARCELID",
            "W7_A": "PARCELID", "W7_B": "PARCELID",
            "W8_A": "PARCELID", "W8_B": "PARCELID",
        },
        {
            "target": "A5AQ3",
            "is_key": True,
            "standard_name": "PLOT_ID",
            "W1_A": "A5AQ3", "W1_B": "A5BQ3",
            "W2_A": "PLTID", "W2_B": "PLTID",
            "W3_A": "PLOTID", "W3_B": "PLOTID",
            "W4_A": "PLOTID", "W4_B": "PLOTID",
            "W5_A": "PLOTID", "W5_B": "PLOTID",
            "W7_A": "PLTID", "W7_B": "PLTID",
            "W8_A": "PLTID", "W8_B": "PLTID",
        },
        {
            "target": "A5AQ5",
            "is_key": True,
            "standard_name": "CROP_ID",
            "W1_A": "A5AQ5", "W1_B": "A5BQ5",
            "W2_A": "CROPID", "W2_B": "CROPID",
            "W3_A": "CROPID", "W3_B": "CROPID",
            "W4_A": "CROPID", "W4_B": "CROPID",
            "W5_A": "CROPID", "W5_B": "CROPID",
            "W7_A": "CROPID", "W7_B": "CROPID",
            "W8_A": "CROPID", "W8_B": "CROPID",
        },
        {
            "target": "A5AQ6A",
            "is_key": False,
            "W1_A": "A5AQ6A", "W1_B": "A5BQ6A",
            "W2_A": "A5AQ6A", "W2_B": "A5BQ6A",
            "W3_A": "A5AQ6A", "W3_B": "A5BQ6A",
            "W4_A": "A5AQ6A", "W4_B": "A5BQ6A",
            "W5_A": "A5AQ6A", "W5_B": "A5BQ6A",
            "W7_A": "S5AQ06A_1", "W7_B": "S5BQ06A_1",
            "W8_A": "S5AQ06A_1", "W8_B": "S5BQ06A_1",
        },
        {
            "target": "A5AQ6B",
            "is_key": False,
            "W1_A": "A5AQ6B", "W1_B": "A5BQ6B",
            "W2_A": "A5AQ6B", "W2_B": "A5BQ6B",
            "W3_A": "A5AQ6B", "W3_B": "A5BQ6B",
            "W4_A": "A5AQ6B", "W4_B": "A5BQ6B",
            "W5_A": "A5AQ6B", "W5_B": "A5BQ6B",
            "W7_A": "A5AQ6B", "W7_B": "A5BQ6B",
            "W8_A": "S5AQ06C_1", "W8_B": "S5BQ06C_1",
        },
        {
            "target": "A5AQ6C",
            "is_key": False,
            "W1_A": "A5AQ6C", "W1_B": "A5BQ6C",
            "W2_A": "A5AQ6C", "W2_B": "A5BQ6C",
            "W3_A": "A5AQ6C", "W3_B": "A5BQ6C",
            "W4_A": "A5AQ6C", "W4_B": "A5BQ6C",
            "W5_A": "A5AQ6C", "W5_B": "A5BQ6C",
            "W7_A": "A5AQ6C", "W7_B": "A5BQ6C",
            "W8_A": "S5AQ06B_1", "W8_B": "S5BQ06B_1",
        },
        {
            "target": "A5AQ6D",
            "is_key": False,
            "W1_A": "A5AQ6D", "W1_B": "A5BQ6D",
            "W2_A": "A5AQ6D", "W2_B": "A5BQ6D",
            "W3_A": "A5AQ6D", "W3_B": "A5BQ6D",
            "W4_A": "A5AQ6D", "W4_B": "A5BQ6D",
            "W5_A": "A5AQ6D", "W5_B": "A5BQ6D",
            "W7_A": "A5AQ6D", "W7_B": "A5BQ6D",
            "W8_A": "S5AQ06D_1", "W8_B": "S5BQ06D_1",
        },
        {
            "target": "A5AQ7A",
            "is_key": False,
            "W1_A": "A5AQ7A", "W1_B": "A5BQ7A",
            "W2_A": "A5AQ7A", "W2_B": "A5BQ7A",
            "W3_A": "A5AQ7A", "W3_B": "A5BQ7A",
            "W4_A": "A5AQ7A", "W4_B": "A5BQ7A",
            "W5_A": "A5AQ7A", "W5_B": "A5BQ7A",
            "W7_A": "S5AQ07A_1", "W7_B": "S5BQ07A_1",
            "W8_A": "S5AQ07A_1", "W8_B": "S5BQ07A_1",
        },
        {
            "target": "A5AQ7B",
            "is_key": False,
            "W1_A": "A5AQ7B", "W1_B": "A5BQ7B",
            "W2_A": "A5AQ7B", "W2_B": "A5BQ7B",
            "W3_A": None, "W3_B": None,
            "W4_A": "A5AQ7B", "W4_B": "A5BQ7B",
            "W5_A": "A5AQ7B", "W5_B": "A5BQ7B",
            "W7_A": "S5AQ07B_1", "W7_B": "S5BQ07B_1",
            "W8_A": "S5AQ07B_1", "W8_B": "S5BQ07B_1",
        },
        {
            "target": "A5AQ7C",
            "is_key": False,
            "W1_A": "A5AQ7C", "W1_B": "A5BQ7C",
            "W2_A": "A5AQ7C", "W2_B": "A5BQ7C",
            "W3_A": "A5AQ7C", "W3_B": "A5BQ7C",
            "W4_A": "A5AQ7C", "W4_B": "A5BQ7C",
            "W5_A": "A5AQ7C", "W5_B": "A5BQ7C",
            "W7_A": "S5AQ07C_1", "W7_B": "S5BQ07C_1",
            "W8_A": "S5AQ07C_1", "W8_B": "S5BQ07C_1",
        },
        {
            "target": "A5AQ8",
            "is_key": False,
            "W1_A": "A5AQ8", "W1_B": "A5BQ8",
            "W2_A": "A5AQ8", "W2_B": "A5BQ8",
            "W3_A": "A5AQ8", "W3_B": "A5BQ8",
            "W4_A": "A5AQ8", "W4_B": "A5BQ8",
            "W5_A": "A5AQ8", "W5_B": "A5BQ8",
            "W7_A": "S5AQ08_1", "W7_B": "S5BQ08_1",
            "W8_A": "S5AQ08_1", "W8_B": "S5BQ08_1",
        },
        {
            "target": "A5AQ15",
            "is_key": False,
            "W1_A": "A5AQ15", "W1_B": "A5BQ15",
            "W2_A": "A5AQ15", "W2_B": "A5BQ15",
            "W3_A": "A5AQ15", "W3_B": "A5BQ15",
            "W4_A": "A5AQ15", "W4_B": "A5BQ15",
            "W5_A": "A5AQ15", "W5_B": "A5BQ15",
            "W7_A": None, "W7_B": "S5BQ15_1",
            "W8_A": "S5AQ15_1", "W8_B": "S5BQ15_1",
        },
        {
            "target": "A5AQ16",
            "is_key": False,
            "W1_A": "A5AQ16", "W1_B": "A5BQ16",
            "W2_A": "A5AQ16", "W2_B": "A5BQ16",
            "W3_A": "A5AQ16", "W3_B": "A5BQ16",
            "W4_A": "A5AQ16", "W4_B": "A5BQ16",
            "W5_A": "A5AQ16", "W5_B": "A5BQ16",
            "W7_A": None, "W7_B": "S5BQ16_1",
            "W8_A": "S5AQ16_1", "W8_B": "S5BQ16_1",
        },
        {
            "target": "A5AQ17",
            "is_key": False,
            "W1_A": "A5AQ17", "W1_B": "A5BQ17",
            "W2_A": "A5AQ17", "W2_B": "A5BQ17",
            "W3_A": "A5AQ17", "W3_B": "A5BQ17",
            "W4_A": "A5AQ17", "W4_B": "A5BQ17",
            "W5_A": "A5AQ17", "W5_B": "A5BQ17",
            "W7_A": None, "W7_B": "S5BQ17_1",
            "W8_A": "S5AQ17_1", "W8_B": "S5BQ17_1",
        },
        {
            "target": "A5AQ18",
            "is_key": False,
            "W1_A": "A5AQ18", "W1_B": "A5BQ18",
            "W2_A": "A5AQ18", "W2_B": "A5BQ18",
            "W3_A": "A5AQ18", "W3_B": "A5BQ18",
            "W4_A": "A5AQ18", "W4_B": "A5BQ18",
            "W5_A": "A5AQ18", "W5_B": "A5BQ18",
            "W7_A": None, "W7_B": "S5BQ18_1",
            "W8_A": "S5AQ18_1", "W8_B": "S5BQ18_1",
        },
        {
            "target": "A5AQ19",
            "is_key": False,
            "W1_A": "A5AQ19", "W1_B": "A5BQ19",
            "W2_A": "A5AQ19", "W2_B": "A5BQ19",
            "W3_A": "A5AQ19", "W3_B": "A5BQ19",
            "W4_A": "A5AQ19", "W4_B": "A5BQ19",
            "W5_A": "A5AQ19", "W5_B": "A5BQ19",
            "W7_A": None, "W7_B": "S5BQ19_1",
            "W8_A": "S5AQ19_1", "W8_B": "S5BQ19_1",
        },
        {
            "target": "A5AQ24",
            "is_key": False,
            "W1_A": "A5AQ24", "W1_B": "A5BQ24",
            "W2_A": "A5AQ24", "W2_B": "A5BQ24",
            "W3_A": "A5AQ22", "W3_B": "A5BQ22",
            "W4_A": "A5AQ22", "W4_B": "A5BQ22",
            "W5_A": "A5AQ22", "W5_B": "A5BQ22",
            "W7_A": "S5AQ22_1", "W7_B": "S5BQ22_1",
            "W8_A": "S5AQ22_1", "W8_B": "S5BQ22_1",
        },
    ],
}


def build_source_variable_map(spec, source):

    source_col = source["source_col"]
    variable_map = {}

    for row in spec["crosswalk_rows"]:
        raw = row.get(source_col)
        if raw is None or str(raw).strip() == "":
            continue

        raw = str(raw).strip().upper()

        if row.get("is_key"):
            target = row["standard_name"].strip().upper()
        else:
            target = row["target"].strip().upper()

        variable_map[raw] = target

    return variable_map


for source in AGSEC5_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(AGSEC5_SPEC, source)


AGSEC5_SPEC["sources"][0]["variable_map"]

In [ ]:
# RUN AGSEC5


agsec5 = stack_section(AGSEC5_SPEC)
export_section(agsec5, "AGSEC5")

agsec5.head()

In [4]:
#AGSEC4 key map


AGSEC4_SPEC = {
    "section": "AGSEC4",
    "level": "crop_plot",
    "standard_keys": ["HHID", "PARCEL_ID", "PLOT_ID", "CROP_ID"],

    "sources": [
        {"wave": 1, "visit": 1, "file": "AGSEC4A.dta", "source_section": "AGSEC4A", "source_col": "W1_A"},
        {"wave": 1, "visit": 2, "file": "AGSEC4B.dta", "source_section": "AGSEC4B", "source_col": "W1_B"},

        {"wave": 2, "visit": 1, "file": "AGSEC4A.dta", "source_section": "AGSEC4A", "source_col": "W2_A"},
        {"wave": 2, "visit": 2, "file": "AGSEC4B.dta", "source_section": "AGSEC4B", "source_col": "W2_B"},

        {"wave": 3, "visit": 1, "file": "AGSEC4A.dta", "source_section": "AGSEC4A", "source_col": "W3_A"},
        {"wave": 3, "visit": 2, "file": "AGSEC4B.dta", "source_section": "AGSEC4B", "source_col": "W3_B"},

        {"wave": 4, "visit": 1, "file": "AGSEC4A.dta", "source_section": "AGSEC4A", "source_col": "W4_A"},
        {"wave": 4, "visit": 2, "file": "AGSEC4B.dta", "source_section": "AGSEC4B", "source_col": "W4_B"},

        {"wave": 5, "visit": 1, "file": "AGSEC4A.dta", "source_section": "AGSEC4A", "source_col": "W5_A"},
        {"wave": 5, "visit": 2, "file": "AGSEC4B.dta", "source_section": "AGSEC4B", "source_col": "W5_B"},

        {"wave": 7, "visit": 1, "file": "AGSEC4A.dta", "source_section": "AGSEC4A", "source_col": "W7_A"},
        {"wave": 7, "visit": 2, "file": "AGSEC4B.dta", "source_section": "AGSEC4B", "source_col": "W7_B"},

        {"wave": 8, "visit": 1, "file": "AGSEC4A.dta", "source_section": "AGSEC4A", "source_col": "W8_A"},
        {"wave": 8, "visit": 2, "file": "AGSEC4B.dta", "source_section": "AGSEC4B", "source_col": "W8_B"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1_A": "HHID", "W1_B": "HHID",
            "W2_A": "HHID", "W2_B": "HHID",
            "W3_A": "HHID", "W3_B": "HHID",
            "W4_A": "HHID", "W4_B": "HHID",
            "W5_A": "HHID", "W5_B": "HHID",
            "W7_A": "HHID", "W7_B": "HHID",
            "W8_A": "HHID", "W8_B": "HHID",
        },
        {
            "target": "A4AQ1",
            "is_key": False,
            "W1_A": "A4AQ1", "W1_B": "A4BQ1",
            "W2_A": None, "W2_B": None,
            "W3_A": None, "W3_B": None,
            "W4_A": None, "W4_B": None,
            "W5_A": None, "W5_B": None,
            "W7_A": None, "W7_B": None,
            "W8_A": None, "W8_B": None,
        },
        {
            "target": "A4AQ2",
            "is_key": True,
            "standard_name": "PARCEL_ID",
            "W1_A": "A4AQ2", "W1_B": "A4BQ2",
            "W2_A": "PRCID", "W2_B": "PRCID",
            "W3_A": "PARCELID", "W3_B": "PARCELID",
            "W4_A": "PARCELID", "W4_B": "PARCELID",
            "W5_A": "PARCELID", "W5_B": "PARCELID",
            "W7_A": "PARCELID", "W7_B": "PARCELID",
            "W8_A": "PARCELID", "W8_B": "PARCELID",
        },
        {
            "target": "A4AQ4",
            "is_key": True,
            "standard_name": "PLOT_ID",
            "W1_A": "A4AQ4", "W1_B": "A4BQ4",
            "W2_A": "PLTID", "W2_B": "PLTID",
            "W3_A": "PLOTID", "W3_B": "PLOTID",
            "W4_A": "PLOTID", "W4_B": "PLOTID",
            "W5_A": "PLOTID", "W5_B": "PLOTID",
            "W7_A": "PLTID", "W7_B": "PLTID",
            "W8_A": "PLTID", "W8_B": "PLTID",
        },
        {
            "target": "A4AQ6",
            "is_key": True,
            "standard_name": "CROP_ID",
            "W1_A": "A4AQ6", "W1_B": "A4BQ6",
            "W2_A": "CROPID", "W2_B": "CROPID",
            "W3_A": "CROPID", "W3_B": "CROPID",
            "W4_A": "CROPID", "W4_B": "CROPID",
            "W5_A": "CROPID", "W5_B": "CROPID",
            "W7_A": "CROPID", "W7_B": "CROPID",
            "W8_A": "CROPID", "W8_B": "CROPID",
        },
        {
            "target": "A4AQ7",
            "is_key": False,
            "W1_A": "A4AQ7", "W1_B": "A4BQ7",
            "W2_A": "A4AQ7", "W2_B": "A4BQ7",
            "W3_A": "A4AQ8", "W3_B": "A4BQ8",
            "W4_A": "A4AQ8", "W4_B": "A4BQ8",
            "W5_A": "A4AQ8", "W5_B": "A4BQ8",
            "W7_A": "S4AQ08", "W7_B": "S4BQ08",
            "W8_A": "S4AQ08", "W8_B": "S4BQ08",
        },
        {
            "target": "A4AQ8",
            "is_key": False,
            "W1_A": "A4AQ8", "W1_B": "A4BQ8",
            "W2_A": "A4AQ8", "W2_B": "A4BQ8",
            "W3_A": "A4AQ7", "W3_B": "A4BQ7",
            "W4_A": "A4AQ7", "W4_B": "A4BQ7",
            "W5_A": "A4AQ7", "W5_B": "A4BQ7",
            "W7_A": "S4AQ07", "W7_B": "S4BQ07",
            "W8_A": "S4AQ07", "W8_B": "S4BQ07",
        },
        {
            "target": "A4AQ9",
            "is_key": False,
            "W1_A": "A4AQ9", "W1_B": "A4BQ9",
            "W2_A": "A4AQ9", "W2_B": "A4BQ9",
            "W3_A": None, "W3_B": None,
            "W4_A": "A4AQ9", "W4_B": "A4BQ9",
            "W5_A": "A4AQ9", "W5_B": "A4BQ9",
            "W7_A": "S4AQ09", "W7_B": "S4BQ09",
            "W8_A": "S4AQ09", "W8_B": "S4BQ09",
        },
        {
            "target": "A4AQ10",
            "is_key": False,
            "W1_A": "A4AQ10", "W1_B": "A4BQ10",
            "W2_A": "A4AQ10", "W2_B": "A4BQ10",
            "W3_A": "A4AQ10", "W3_B": "A4BQ10",
            "W4_A": "A4AQ16", "W4_B": "A4BQ10",
            "W5_A": "A4AQ10", "W5_B": "A4BQ10",
            "W7_A": "S4AQ10", "W7_B": "S4BQ10",
            "W8_A": "S4AQ10", "W8_B": "S4BQ10",
        },
        {
            "target": "A4AQ11",
            "is_key": False,
            "W1_A": "A4AQ11", "W1_B": "A4BQ11",
            "W2_A": "A4AQ11", "W2_B": "A4BQ11",
            "W3_A": "A4AQ15", "W3_B": "A4BQ15",
            "W4_A": "A4AQ15", "W4_B": "A4BQ15",
            "W5_A": "A4AQ15", "W5_B": "A4BQ15",
            "W7_A": "S4AQ15", "W7_B": "S4BQ15",
            "W8_A": "S4AQ15", "W8_B": "S4BQ15",
        },
        {
            "target": "A4AQ12",
            "is_key": False,
            "W1_A": "A4AQ12", "W1_B": "A4BQ12",
            "W2_A": "A4AQ12", "W2_B": "A4BQ12",
            "W3_A": None, "W3_B": None,
            "W4_A": None, "W4_B": None,
            "W5_A": None, "W5_B": None,
            "W7_A": None, "W7_B": None,
            "W8_A": None, "W8_B": None,
        },
        {
            "target": "A4AQ13",
            "is_key": False,
            "W1_A": "A4AQ13", "W1_B": "A4BQ13",
            "W2_A": "A4AQ13", "W2_B": "A4BQ13",
            "W3_A": "A4AQ13", "W3_B": "A4BQ13",
            "W4_A": "A4AQ13", "W4_B": "A4BQ13",
            "W5_A": "A4AQ13", "W5_B": "A4BQ13",
            "W7_A": "S4AQ13", "W7_B": "S4BQ13",
            "W8_A": "S4AQ13", "W8_B": "S4BQ13",
        },
        {
            "target": "A4AQ14",
            "is_key": False,
            "W1_A": "A4AQ14", "W1_B": "A4BQ14",
            "W2_A": "A4AQ14", "W2_B": "A4BQ14",
            "W3_A": "A4AQ14", "W3_B": "A4BQ14",
            "W4_A": "A4AQ14", "W4_B": "A4BQ14",
            "W5_A": "A4AQ14", "W5_B": "A4BQ14",
            "W7_A": "S4AQ14", "W7_B": "S4BQ14",
            "W8_A": "S4AQ14", "W8_B": "S4BQ14",
        },
    ],
}

for source in AGSEC4_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(AGSEC4_SPEC, source)

agsec4 = stack_section(AGSEC4_SPEC)
export_section(agsec4, "AGSEC4")

agsec4.head()


Loaded Wave 1, Visit 1: AGSEC4A.dta
Rows: 13,987
Mapped variables found: 13/13

Loaded Wave 1, Visit 2: AGSEC4B.dta
Rows: 13,422
Mapped variables found: 13/13

Loaded Wave 2, Visit 1: AGSEC4A.dta
Rows: 12,191
Mapped variables found: 12/12

Loaded Wave 2, Visit 2: AGSEC4B.dta
Rows: 9,878
Mapped variables found: 12/12

Loaded Wave 3, Visit 1: AGSEC4A.dta
Rows: 10,938
Mapped variables found: 10/10

Loaded Wave 3, Visit 2: AGSEC4B.dta
Rows: 9,060
Mapped variables found: 10/10

Loaded Wave 4, Visit 1: AGSEC4A.dta
Rows: 10,550
Mapped variables found: 11/11

Loaded Wave 4, Visit 2: AGSEC4B.dta
Rows: 9,146
Mapped variables found: 11/11

Loaded Wave 5, Visit 1: AGSEC4A.dta
Rows: 10,823
Mapped variables found: 11/11

Loaded Wave 5, Visit 2: AGSEC4B.dta
Rows: 9,691
Mapped variables found: 11/11

Loaded Wave 7, Visit 1: AGSEC4A.dta
Rows: 8,776
Mapped variables found: 11/11

Loaded Wave 7, Visit 2: AGSEC4B.dta
Rows: 7,063
Mapped variables found: 11/11

Loaded Wave 8, Visit 1: AGSEC4A.dta
Rows: 8,4

,WAVE,VISIT,SOURCE_FILE,SOURCE_SECTION,HHID,A4AQ1,PARCEL_ID,PLOT_ID,CROP_ID,A4AQ7,A4AQ8,A4AQ9,A4AQ10,A4AQ11,A4AQ12,A4AQ13,A4AQ14
0,1,1,AGSEC4A.dta,AGSEC4A,1013000204,NaN,21.0,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,AGSEC4A.dta,AGSEC4A,1021000108,1.0,1.0,1.0,630.0,2.0,0.00,NaN,2.0,NaN,NaN,NaN,NaN
2,1,1,AGSEC4A.dta,AGSEC4A,1021000108,1.0,1.0,1.0,741.0,2.0,0.50,NaN,2.0,NaN,NaN,NaN,NaN
3,1,1,AGSEC4A.dta,AGSEC4A,1021000113,1.0,1.0,1.0,130.0,2.0,0.25,5.0,2.0,NaN,NaN,NaN,NaN
4,1,1,AGSEC4A.dta,AGSEC4A,1021000113,1.0,1.0,1.0,210.0,2.0,0.25,10.0,1.0,1600.0,3.0,1.0,NaN


In [4]:
# AGSEC3A/B SPEC

def build_source_variable_map(spec, source):

    source_col = source["source_col"]
    variable_map = {}

    for row in spec["crosswalk_rows"]:
        raw = row.get(source_col)

        if raw is None or str(raw).strip() == "":
            continue

        raw = str(raw).strip().upper()

        if row.get("is_key"):
            target = row["standard_name"].strip().upper()
        else:
            target = row["target"].strip().upper()

        variable_map[raw] = target

    return variable_map


AGSEC3_SPEC = {
    "section": "AGSEC3",
    "level": "plot",
    "standard_keys": ["HHID", "PARCEL_ID", "PLOT_ID"],

    "sources": [
        {"wave": 1, "visit": 1, "file": "AGSEC3A.dta", "source_section": "AGSEC3A", "source_col": "W1_A"},
        {"wave": 1, "visit": 2, "file": "AGSEC3B.dta", "source_section": "AGSEC3B", "source_col": "W1_B"},

        {"wave": 2, "visit": 1, "file": "AGSEC3A.dta", "source_section": "AGSEC3A", "source_col": "W2_A"},
        {"wave": 2, "visit": 2, "file": "AGSEC3B.dta", "source_section": "AGSEC3B", "source_col": "W2_B"},

        {"wave": 3, "visit": 1, "file": "AGSEC3A.dta", "source_section": "AGSEC3A", "source_col": "W3_A"},
        {"wave": 3, "visit": 2, "file": "AGSEC3B.dta", "source_section": "AGSEC3B", "source_col": "W3_B"},

        {"wave": 4, "visit": 1, "file": "AGSEC3A.dta", "source_section": "AGSEC3A", "source_col": "W4_A"},
        {"wave": 4, "visit": 2, "file": "AGSEC3B.dta", "source_section": "AGSEC3B", "source_col": "W4_B"},

        {"wave": 5, "visit": 1, "file": "AGSEC3A.dta", "source_section": "AGSEC3A", "source_col": "W5_A"},
        {"wave": 5, "visit": 2, "file": "AGSEC3B.dta", "source_section": "AGSEC3B", "source_col": "W5_B"},

        {"wave": 7, "visit": 1, "file": "AGSEC3A.dta", "source_section": "AGSEC3A", "source_col": "W7_A"},
        {"wave": 7, "visit": 2, "file": "AGSEC3B.dta", "source_section": "AGSEC3B", "source_col": "W7_B"},

        {"wave": 8, "visit": 1, "file": "AGSEC3A.dta", "source_section": "AGSEC3A", "source_col": "W8_A"},
        {"wave": 8, "visit": 2, "file": "AGSEC3B.dta", "source_section": "AGSEC3B", "source_col": "W8_B"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1_A": "HHID", "W1_B": "HHID",
            "W2_A": "HHID", "W2_B": "HHID",
            "W3_A": "HHID", "W3_B": "HHID",
            "W4_A": "HHID", "W4_B": "HHID",
            "W5_A": "HHID", "W5_B": "HHID",
            "W7_A": "HHID", "W7_B": "HHID",
            "W8_A": "HHID", "W8_B": "HHID",
        },
        {
            "target": "A3AQ1",
            "is_key": True,
            "standard_name": "PARCEL_ID",
            "W1_A": "A3AQ1", "W1_B": "A3BQ1",
            "W2_A": "PRCID", "W2_B": "PRCID",
            "W3_A": "PARCELID", "W3_B": "PARCELID",
            "W4_A": "PARCELID", "W4_B": "PARCELID",
            "W5_A": "PARCELID", "W5_B": "PARCELID",
            "W7_A": "PARCELID", "W7_B": "PARCELID",
            "W8_A": "PARCELID", "W8_B": "PARCELID",
        },
        {
            "target": "A3AQ3",
            "is_key": True,
            "standard_name": "PLOT_ID",
            "W1_A": "A3AQ3", "W1_B": "A3BQ3",
            "W2_A": "PLTID", "W2_B": "PLTID",
            "W3_A": "PLOTID", "W3_B": "PLOTID",
            "W4_A": "PLOTID", "W4_B": "PLOTID",
            "W5_A": "PLOTID", "W5_B": "PLOTID",
            "W7_A": "PLTID", "W7_B": "PLTID",
            "W8_A": "PLTID", "W8_B": "PLTID",
        },
        {
            "target": "A3AQ4",
            "is_key": False,
            "W1_A": "A3AQ4", "W1_B": "A3BQ4",
            "W2_A": "A3AQ4", "W2_B": "A3BQ4",
            "W3_A": "A3AQ4", "W3_B": "A3BQ4",
            "W4_A": "A3AQ4", "W4_B": "A3BQ4",
            "W5_A": "A3AQ4", "W5_B": "A3BQ4",
            "W7_A": "S3AQ04", "W7_B": "S3BQ04",
            "W8_A": "S3AQ04", "W8_B": "S3BQ04",
        },
        {
            "target": "A3AQ14",
            "is_key": False,
            "W1_A": "A3AQ14", "W1_B": "A3BQ14",
            "W2_A": "A3AQ14", "W2_B": "A3BQ14",
            "W3_A": "A3AQ13", "W3_B": "A3BQ13",
            "W4_A": "A3AQ13", "W4_B": "A3BQ13",
            "W5_A": "A3AQ13", "W5_B": "A3BQ13",
            "W7_A": "S3AQ13", "W7_B": "S3BQ13",
            "W8_A": "S3AQ13", "W8_B": "S3BQ13",
        },
        {
            "target": "A3AQ26",
            "is_key": False,
            "W1_A": "A3AQ26", "W1_B": "A3BQ26",
            "W2_A": "A3AQ26", "W2_B": "A3BQ26",
            "W3_A": "A3AQ22", "W3_B": "A3BQ22",
            "W4_A": "A3AQ22", "W4_B": "A3BQ22",
            "W5_A": "A3AQ22", "W5_B": "A3BQ22",
            "W7_A": "S3AQ22", "W7_B": "S3BQ22",
            "W8_A": "S3AQ22", "W8_B": "S3BQ22",
        },
        {
            "target": "A3AQ38",
            "is_key": False,
            "W1_A": "A3AQ38", "W1_B": "A3BQ38",
            "W2_A": "A3AQ38", "W2_B": "A3BQ38",
            "W3_A": "A3AQ31", "W3_B": "A3BQ31",
            "W4_A": "AGG_FAM_LABOUR_COUNT", "W4_B": "AGG_FAM_LABOUR_COUNT",
            "W5_A": "AGG_FAM_LABOUR_COUNT", "W5_B": "AGG_FAM_LABOUR_COUNT",
            "W7_A": None, "W7_B": None,
            "W8_A": "AGG_FAM_LABOUR_COUNT", "W8_B": "AGG_FAM_LABOUR_COUNT",
        },
        {
            "target": "A3AQ39",
            "is_key": False,
            "W1_A": "A3AQ39", "W1_B": "A3BQ39",
            "W2_A": "A3AQ39", "W2_B": "A3BQ39",
            "W3_A": "A3AQ32", "W3_B": "A3BQ32",
            "W4_A": "AGG_FAM_LABOUR_DAYS", "W4_B": "AGG_FAM_LABOUR_DAYS",
            "W5_A": "AGG_FAM_LABOUR_DAYS", "W5_B": "AGG_FAM_LABOUR_DAYS",
            "W7_A": None, "W7_B": None,
            "W8_A": "AGG_FAM_LABOUR_DAYS", "W8_B": "AGG_FAM_LABOUR_DAYS",
        },
        {
            "target": "A3AQ41",
            "is_key": False,
            "W1_A": "A3AQ41", "W1_B": "A3BQ41",
            "W2_A": "A3AQ41", "W2_B": "A3BQ41",
            "W3_A": "A3AQ34", "W3_B": "A3BQ34",
            "W4_A": "A3AQ34", "W4_B": "A3BQ34",
            "W5_A": "A3AQ34", "W5_B": "A3BQ34",
            "W7_A": "S3AQ34", "W7_B": None,
            "W8_A": "S3AQ34", "W8_B": "S3BQ34",
        },
        {
            "target": "A3AQ43",
            "is_key": False,
            "W1_A": "A3AQ43", "W1_B": "A3BQ43",
            "W2_A": "A3AQ43", "W2_B": "A3BQ43",
            "W3_A": "A3AQ36", "W3_B": "A3BQ36",
            "W4_A": "A3AQ36", "W4_B": "A3BQ36",
            "W5_A": "A3AQ36", "W5_B": "A3BQ36",
            "W7_A": "S3AQ36", "W7_B": None,
            "W8_A": "S3AQ36", "W8_B": "S3BQ36",
        },
    ],
}



# The labour-count/day variables come from AGSEC3A_1/B_1.dta.


for source in AGSEC3_SPEC["sources"]:
    if source["wave"] == 8 and source["visit"] == 1:
        source["labor_file"] = "AGSEC3A_1.dta"
    if source["wave"] == 8 and source["visit"] == 2:
        source["labor_file"] = "AGSEC3B_1.dta"



In [3]:
#AGSEC3 out

for source in AGSEC3_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(AGSEC3_SPEC, source)

agsec3 = stack_section(AGSEC3_SPEC)
export_section(agsec3, "AGSEC3")

agsec3.head()


Loaded Wave 1, Visit 1: AGSEC3A.dta
Rows: 9,898
Mapped variables found: 10/10

Loaded Wave 1, Visit 2: AGSEC3B.dta
Rows: 9,789
Mapped variables found: 10/10

Loaded Wave 2, Visit 1: AGSEC3A.dta
Rows: 8,107
Mapped variables found: 10/10

Loaded Wave 2, Visit 2: AGSEC3B.dta
Rows: 6,911
Mapped variables found: 10/10

Loaded Wave 3, Visit 1: AGSEC3A.dta
Rows: 7,529
Mapped variables found: 10/10

Loaded Wave 3, Visit 2: AGSEC3B.dta
Rows: 6,500
Mapped variables found: 10/10

Loaded Wave 4, Visit 1: AGSEC3A.dta
Rows: 7,550
Mapped variables found: 10/10

Loaded Wave 4, Visit 2: AGSEC3B.dta
Rows: 6,756
Mapped variables found: 10/10

Loaded Wave 5, Visit 1: AGSEC3A.dta
Rows: 7,787
Mapped variables found: 10/10

Loaded Wave 5, Visit 2: AGSEC3B.dta
Rows: 6,792
Mapped variables found: 10/10

Loaded Wave 7, Visit 1: AGSEC3A.dta
Rows: 6,073
Mapped variables found: 8/8

Loaded Wave 7, Visit 2: AGSEC3B.dta
Rows: 6,134
Mapped variables found: 6/6

Loaded Wave 8, Visit 1: AGSEC3A.dta
Rows: 5,884
Mapped 

,WAVE,VISIT,SOURCE_FILE,SOURCE_SECTION,HHID,PARCEL_ID,PLOT_ID,A3AQ4,A3AQ14,A3AQ26,A3AQ38,A3AQ39,A3AQ41,A3AQ43
0,1,1,AGSEC3A.dta,AGSEC3A,1013000204,21.0,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,AGSEC3A.dta,AGSEC3A,1021000108,1.0,1.0,1.0,2.0,2.0,1.0,28.0,1.0,30000.0
2,1,1,AGSEC3A.dta,AGSEC3A,1021000113,1.0,1.0,1.0,2.0,2.0,2.0,36.0,1.0,50000.0
3,1,1,AGSEC3A.dta,AGSEC3A,1021000408,1.0,1.0,2.0,2.0,2.0,1.0,2.0,2.0,NaN
4,1,1,AGSEC3A.dta,AGSEC3A,1021000408,1.0,2.0,2.0,2.0,2.0,1.0,30.0,2.0,NaN


In [8]:
#AGSEC2 key map

AGSEC2_SPEC = {
    "section": "AGSEC2",
    "level": "parcel",
    "standard_keys": ["HHID", "PARCEL_ID"],
    "metadata_cols": ["OWNERSHIP"],

    "sources": [
        {"wave": 1, "ownership": "self_owned", "file": "AGSEC2A.dta", "source_section": "AGSEC2A", "source_col": "W1_A"},
        {"wave": 1, "ownership": "use_rights", "file": "AGSEC2B.dta", "source_section": "AGSEC2B", "source_col": "W1_B"},

        {"wave": 2, "ownership": "self_owned", "file": "AGSEC2A.dta", "source_section": "AGSEC2A", "source_col": "W2_A"},
        {"wave": 2, "ownership": "use_rights", "file": "AGSEC2B.dta", "source_section": "AGSEC2B", "source_col": "W2_B"},

        {"wave": 3, "ownership": "self_owned", "file": "AGSEC2A.dta", "source_section": "AGSEC2A", "source_col": "W3_A"},
        {"wave": 3, "ownership": "use_rights", "file": "AGSEC2B.dta", "source_section": "AGSEC2B", "source_col": "W3_B"},

        {"wave": 4, "ownership": "self_owned", "file": "AGSEC2A.dta", "source_section": "AGSEC2A", "source_col": "W4_A"},
        {"wave": 4, "ownership": "use_rights", "file": "AGSEC2B.dta", "source_section": "AGSEC2B", "source_col": "W4_B"},

        {"wave": 5, "ownership": "self_owned", "file": "AGSEC2A.dta", "source_section": "AGSEC2A", "source_col": "W5_A"},
        {"wave": 5, "ownership": "use_rights", "file": "AGSEC2B.dta", "source_section": "AGSEC2B", "source_col": "W5_B"},

        {"wave": 7, "ownership": "self_owned", "file": "AGSEC2A.dta", "source_section": "AGSEC2A", "source_col": "W7_A"},
        {"wave": 7, "ownership": "use_rights", "file": "AGSEC2B.dta", "source_section": "AGSEC2B", "source_col": "W7_B"},

        {"wave": 8, "ownership": "self_owned", "file": "AGSEC2A.dta", "source_section": "AGSEC2A", "source_col": "W8_A"},
        {"wave": 8, "ownership": "use_rights", "file": "AGSEC2B.dta", "source_section": "AGSEC2B", "source_col": "W8_B"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1_A": "HHID", "W1_B": "HHID",
            "W2_A": "HHID", "W2_B": "HHID",
            "W3_A": "HHID", "W3_B": "HHID",
            "W4_A": "HHID", "W4_B": "HHID",
            "W5_A": "HHID", "W5_B": "HHID",
            "W7_A": "HHID", "W7_B": "HHID",
            "W8_A": "HHID", "W8_B": "HHID",
        },
        {
            "target": "A2Q2",
            "is_key": True,
            "standard_name": "PARCEL_ID",
            "W1_A": "A2AQ2", "W1_B": "A2BQ2",
            "W2_A": "PRCID", "W2_B": "PRCID",
            "W3_A": "PARCELID", "W3_B": "PARCELID",
            "W4_A": "PARCELID", "W4_B": "PARCELID",
            "W5_A": "PARCELID", "W5_B": "PARCELID",
            "W7_A": "PARCELID", "W7_B": "PARCELID",
            "W8_A": "PARCELID", "W8_B": "PARCELID",
        },
        {
            "target": "A2Q4",
            "is_key": False,
            "W1_A": "A2AQ4", "W1_B": "A2BQ4",
            "W2_A": "A2AQ4", "W2_B": "A2BQ4",
            "W3_A": "A2AQ4", "W3_B": "A2BQ4",
            "W4_A": "A2AQ4", "W4_B": "A2BQ4",
            "W5_A": "A2AQ4", "W5_B": "A2BQ4",
            "W7_A": "S2AQ4", "W7_B": "S2AQ04",
            "W8_A": "S2AQ4", "W8_B": "S2AQ04",
        },
        {
            "target": "A2Q5",
            "is_key": False,
            "W1_A": "A2AQ5", "W1_B": "A2BQ5",
            "W2_A": "A2AQ5", "W2_B": "A2BQ5",
            "W3_A": "A2AQ5", "W3_B": "A2BQ5",
            "W4_A": "A2AQ5", "W4_B": "A2BQ5",
            "W5_A": "A2AQ5", "W5_B": "A2BQ5",
            "W7_A": "S2AQ5", "W7_B": "S2AQ05",
            "W8_A": "S2AQ5", "W8_B": "S2AQ05",
        },
        {
            "target": "A2Q13A",
            "is_key": False,
            "W1_A": "A2AQ13A", "W1_B": "A2BQ15A",
            "W2_A": "A2AQ13A", "W2_B": "A2BQ15A",
            "W3_A": "A2AQ11A", "W3_B": "A2BQ12A",
            "W4_A": "A2AQ11A", "W4_B": "A2BQ12A",
            "W5_A": "A2AQ11A", "W5_B": "A2BQ12A",
            "W7_A": "S2AQ11A", "W7_B": "A2BQ12A",
            "W8_A": "S2AQ11A", "W8_B": "A2BQ12A",
        },
        {
            "target": "A2Q13B",
            "is_key": False,
            "W1_A": "A2AQ13B", "W1_B": "A2BQ15B",
            "W2_A": "A2AQ13B", "W2_B": "A2BQ15B",
            "W3_A": "A2AQ11B", "W3_B": "A2BQ12B",
            "W4_A": "A2AQ11B", "W4_B": "A2BQ12B",
            "W5_A": "A2AQ11B", "W5_B": "A2BQ12B",
            "W7_A": "S2AQ11B", "W7_B": "A2BQ12B",
            "W8_A": "S2AQ11B", "W8_B": "A2BQ12B",
        },
        {
            "target": "A2Q17A",
            "is_key": False,
            "W1_A": "A2AQ17A", "W1_B": None,
            "W2_A": "A2AQ17A", "W2_B": None,
            "W3_A": "A2AQ15A", "W3_B": None,
            "W4_A": "A2AQ15A", "W4_B": None,
            "W5_A": "A2AQ15A", "W5_B": None,
            "W7_A": "S2AQ15A", "W7_B": None,
            "W8_A": "S2AQ15A", "W8_B": None,
        },
        {
            "target": "A2Q17B",
            "is_key": False,
            "W1_A": "A2AQ17B", "W1_B": None,
            "W2_A": "A2AQ17B", "W2_B": None,
            "W3_A": "A2AQ15B", "W3_B": None,
            "W4_A": "A2AQ15B", "W4_B": None,
            "W5_A": "A2AQ15B", "W5_B": None,
            "W7_A": "S2AQ15B", "W7_B": None,
            "W8_A": "S2AQ15B", "W8_B": None,
        },
        {
            "target": "A2Q18",
            "is_key": False,
            "W1_A": "A2AQ18", "W1_B": "A2BQ17",
            "W2_A": "A2AQ18", "W2_B": "A2BQ17",
            "W3_A": "A2AQ16", "W3_B": "A2BQ14",
            "W4_A": "A2AQ16", "W4_B": "A2BQ14",
            "W5_A": "A2AQ16", "W5_B": "A2BQ14",
            "W7_A": "S2AQ16", "W7_B": "S2AQ16",
            "W8_A": None, "W8_B": None,
        },
        {
            "target": "A2Q19",
            "is_key": False,
            "W1_A": "A2AQ19", "W1_B": "A2BQ18",
            "W2_A": "A2AQ19", "W2_B": "A2BQ18",
            "W3_A": "A2AQ17", "W3_B": "A2BQ15",
            "W4_A": "A2AQ17", "W4_B": "A2BQ15",
            "W5_A": "A2AQ17", "W5_B": "A2BQ15",
            "W7_A": "S2AQ17", "W7_B": "S2AQ17",
            "W8_A": "A2AQ18", "W8_B": "A2AQ18",
        },
        {
            "target": "A2Q20",
            "is_key": False,
            "W1_A": "A2AQ20", "W1_B": "A2BQ19",
            "W2_A": "A2AQ20", "W2_B": "A2BQ19",
            "W3_A": "A2AQ18", "W3_B": "A2BQ16",
            "W4_A": "A2AQ18", "W4_B": "A2BQ16",
            "W5_A": "A2AQ18", "W5_B": "A2BQ16",
            "W7_A": "A2AQ18", "W7_B": "A2AQ18",
            "W8_A": None, "W8_B": None,
        },
        {
            "target": "A2Q7",
            "is_key": False,
            "W1_A": "A2AQ7", "W1_B": "A2BQ7",
            "W2_A": None, "W2_B": "A2BQ7",
            "W3_A": "A2AQ7", "W3_B": "A2BQ7",
            "W4_A": "A2AQ7", "W4_B": "A2BQ7",
            "W5_A": "A2AQ7", "W5_B": "A2BQ7",
            "W7_A": "S2AQ7", "W7_B": "S2AQ07",
            "W8_A": "S2AQ7", "W8_B": "S2AQ07",
        },
    ],
}


In [9]:
#AGSEC2 out mod

for source in AGSEC2_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(AGSEC2_SPEC, source)

agsec2 = stack_section(AGSEC2_SPEC)
export_section(agsec2, "AGSEC2")

agsec2.head()


Loaded Wave 1: AGSEC2A.dta
Source section: AGSEC2A
Ownership: self_owned
Rows: 4,305
Mapped variables found: 12/12

Loaded Wave 1: AGSEC2B.dta
Source section: AGSEC2B
Ownership: use_rights
Rows: 1,520
Mapped variables found: 10/10

Loaded Wave 2: AGSEC2A.dta
Source section: AGSEC2A
Ownership: self_owned
Rows: 3,457
Mapped variables found: 11/11

Loaded Wave 2: AGSEC2B.dta
Source section: AGSEC2B
Ownership: use_rights
Rows: 1,063
Mapped variables found: 10/10

Loaded Wave 3: AGSEC2A.dta
Source section: AGSEC2A
Ownership: self_owned
Rows: 3,763
Mapped variables found: 12/12

Loaded Wave 3: AGSEC2B.dta
Source section: AGSEC2B
Ownership: use_rights
Rows: 1,077
Mapped variables found: 10/10

Loaded Wave 4: AGSEC2A.dta
Source section: AGSEC2A
Ownership: self_owned
Rows: 4,142
Mapped variables found: 12/12

Loaded Wave 4: AGSEC2B.dta
Source section: AGSEC2B
Ownership: use_rights
Rows: 1,294
Mapped variables found: 10/10

Loaded Wave 5: AGSEC2A.dta
Source section: AGSEC2A
Ownership: self_owne

,WAVE,SOURCE_FILE,SOURCE_SECTION,OWNERSHIP,HHID,PARCEL_ID,A2Q4,A2Q5,A2Q13A,A2Q13B,A2Q17A,A2Q17B,A2Q18,A2Q19,A2Q20,A2Q7
0,1,AGSEC2A.dta,AGSEC2A,self_owned,1021000108,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,AGSEC2A.dta,AGSEC2A,self_owned,1021000113,1,NaN,0.50,1.0,1.0,100.0,100.0,2.0,2.0,3.0,6.0
2,1,AGSEC2A.dta,AGSEC2A,self_owned,1021000408,1,NaN,0.50,1.0,1.0,60.0,NaN,3.0,1.0,2.0,3.0
3,1,AGSEC2A.dta,AGSEC2A,self_owned,1021000710,1,1.15,0.25,1.0,NaN,100.0,NaN,1.0,2.0,2.0,3.0
4,1,AGSEC2A.dta,AGSEC2A,self_owned,1021000807,1,NaN,4.00,2.0,2.0,50.0,50.0,2.0,1.0,2.0,3.0
